In [0]:
import requests
import json
import boto3

In [0]:
%run ../helper_function/helper_function

In [0]:
access_key = dbutils.secrets.get(scope="barcakings-secrets",key="s3-access-key")
secret_key = dbutils.secrets.get(scope="barcakings-secrets",key="s3-secret-key")
databricks_token = dbutils.secrets.get(scope="barcakings-secrets",key="dbk-pat")

In [0]:
dbutils.widgets.text("sport", "football", "sport")
sport = dbutils.widgets.get("sport")

# dbutils.widgets.text("file_name", "fcb_players_stats", "file_name")
# file_name = dbutils.widgets.get("file_name")

In [0]:
# Read control table
control_df = spark.sql(
    f"""
        SELECT 
            *
        FROM  
            workspace.bk_raw.raw_control_tbl 
        WHERE 
            sport = '{sport}' 
            -- AND  file_name = '{file_name}' 
    """)
control_df.display()

In [0]:
control_rows = control_df.collect()

for row in control_rows:
    team = row["team"]
    api_url = row["api_url"]
    api_param = json.loads(row["api_param"])
    pn_flag = row["pn_flag"]
    offset = row["offset"]
    limit = row["limit"]
    volume_path = row["target_volume_path"]+file_name+".json"
    bucket_name = row["target_s3_bucket"]
    s3_key = row["target_s3_key"]+file_name+".json"

databricks_host = "https://dbc-934e64d2-fbc7.cloud.databricks.com/"

headers = {
    "User-Agent": "Mozilla/5.0"
}

In [0]:
if pn_flag:
    json_data = pn_api_extraction(api_url,api_param,headers,offset,limit)
else:
    json_data = api_extraction(api_url,api_param)

In [0]:
# Create S3 client
s3 = boto3.client(
    "s3",
    aws_access_key_id=access_key,
    aws_secret_access_key=secret_key,
    region_name="us-east-1"
)

# Upload to S3
s3.put_object(
    Bucket=bucket_name,
    Key=s3_key,
    Body=json_data,
    ContentType="application/json"
)

print("File uploaded to S3 successfully")

response = requests.put(
    f"{databricks_host}/api/2.0/fs/files{volume_path}",
    headers={
        "Authorization": f"Bearer {databricks_token}",
        "Content-Type": "application/octet-stream"
    },
    data=json_data.encode("utf-8")
)

print(response.status_code)
print(response.text)

print("File uploaded to Databricks successfully")